## TRAINING DIFFERENT MODELS

### (1) IMPORTS

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

### (2) LOAD DATA

In [2]:
data = pd.read_csv("../data/chat/chat_cleaned.csv")
X = np.load("../data/chat/chat_features.npy")
y = data["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### (3) TRAINING MODELS

#### Linear Regression

In [3]:
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

preds_lr = model_lr.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, preds_lr))
r2_lr = r2_score(y_test, preds_lr)

print("Linear Regression RMSE:", rmse_lr)
print("Linear Regression R2:", r2_lr)

joblib.dump(model_lr, "../models/linear_regression.pkl")

Linear Regression RMSE: 0.09792671983468693
Linear Regression R2: 0.6609490159905214


['../models/linear_regression.pkl']

#### Ridge Regression

In [5]:
model_ridge = Ridge(alpha=1.0)
model_ridge.fit(X_train, y_train)

preds_ridge = model_ridge.predict(X_test)

rmse_ridge = np.sqrt(mean_squared_error(y_test, preds_ridge))
r2_ridge = r2_score(y_test, preds_ridge)

print("Ridge RMSE:", rmse_ridge)
print("Ridge R2:", r2_ridge)

joblib.dump(model_ridge, "../models/ridge.pkl")

Ridge RMSE: 0.09792297009572988
Ridge R2: 0.6609749808823191


['../models/ridge.pkl']

#### Lasso Regression

In [6]:
model_lasso = Lasso(alpha=0.1)
model_lasso.fit(X_train, y_train)

preds_lasso = model_lasso.predict(X_test)

rmse_lasso = np.sqrt(mean_squared_error(y_test, preds_lasso))
r2_lasso = r2_score(y_test, preds_lasso)

print("Lasso RMSE:", rmse_lasso)
print("Lasso R2:", r2_lasso)

joblib.dump(model_lasso, "../models/lasso.pkl")

Lasso RMSE: 0.1681829579770383
Lasso R2: -6.117474875444451e-05


['../models/lasso.pkl']

#### Random Forest

In [7]:
model_rf = RandomForestRegressor(
    n_estimators=50,   # reduce trees
    max_depth=15,      # limit tree depth
    n_jobs=-1,         # use all CPU cores
    random_state=42
)
model_rf.fit(X_train, y_train)

preds_rf = model_rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, preds_rf))
r2_rf = r2_score(y_test, preds_rf)

print("Random Forest RMSE:", rmse_rf)
print("Random Forest R2:", r2_rf)

joblib.dump(model_rf, "../models/random_forest.pkl")

Random Forest RMSE: 0.09630583015454776
Random Forest R2: 0.6720801154437875


['../models/random_forest.pkl']

#### XGBoost

In [14]:
# Install if not already installed
# !pip install xgboost

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import joblib

# Initialize model
model_xgb = XGBRegressor(
    objective="reg:logistic",
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

# Train
model_xgb.fit(X_train, y_train)

# Predict
preds_xgb = model_xgb.predict(X_test)

# Evaluate
rmse_xgb = np.sqrt(mean_squared_error(y_test, preds_xgb))
r2_xgb = r2_score(y_test, preds_xgb)

print("XGBoost RMSE:", rmse_xgb)
print("XGBoost R2:", r2_xgb)

# Save model
joblib.dump(model_xgb, "../models/xgboost_model.pkl")

XGBoost RMSE: 0.09642860142373036
XGBoost R2: 0.6712435139285231


['../models/xgboost_model.pkl']

### (4) COMPARING MODELS

In [15]:
results = {
    "Linear": (rmse_lr, r2_lr),
    "Ridge": (rmse_ridge, r2_ridge),
    "Lasso": (rmse_lasso, r2_lasso),
    "Random Forest": (rmse_rf, r2_rf),
    "XGBoost":(rmse_xgb,r2_xgb)
}

results_df = pd.DataFrame(results, index=["RMSE", "R2"]).T
results_df.sort_values(by="RMSE")

,RMSE,R2
Random Forest,0.096306,0.672080
XGBoost,0.096429,0.671244
Ridge,0.097923,0.660975
Linear,0.097927,0.660949
Lasso,0.168183,-0.000061


In [25]:
def clean_text(text):
    import re
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer

    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()

    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\d+", "", text)

    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

### (5) EXAMPLE PREDICTION

In [29]:
vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")

chat = ["hell no"]

cleaned_chat = [clean_text(chat[0])]

features = vectorizer.transform(cleaned_chat)

prediction = model_xgb.predict(features)

print("Prediction:", prediction[0])

print(features.toarray())

Prediction: 0.83881515
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0